# 🎯 Prompt Engineering - Day 2 Opener

Welcome to Day 2! Today I'm going to show you three techniques that turn a
vague, unreliable prompt into a precision tool.

## Learning Objectives

By the end of this topic you will be able to:

1. Write few-shot prompts that guide the model with labeled examples
2. Use `json_schema` structured outputs to guarantee field names and value types in every response
3. Run a systematic test battery and score your prompt to drive iterative improvement

## What we are building today

The **intent classification layer** of the Barclays Product Knowledge Assistant.

A customer service agent receives a message like:
`"I need help understanding my recent direct debit."`

Before routing that to the right team, we need a reliable classifier that maps
it to one of five categories:

- `account_inquiry`
- `card_services`
- `loans`
- `investments`
- `general`

By end of this topic:

```
BEFORE -> model guesses differently every time, returns free text with no guaranteed format
AFTER  -> model always returns {"intent": "account_inquiry", "confidence": "high", "rationale": "..."}
```

The `classify_intent()` and `classify_with_schema()` functions you build today
are called directly by Topic 5 (Conversation Memory) to route queries.

## Section 0: Environment Setup

Installing dependencies and connecting to SageMaker and the OpenAI API.

In [ ]:
# openai 1.51.0: first version with fully stable json_schema strict mode
# numpy<2 pinned to avoid SageMaker compatibility issues
!pip install -q "openai==1.51.0" "numpy<2"

In [ ]:
import os
import json
import getpass

from openai import OpenAI

import sagemaker
from sagemaker import get_execution_role

# SageMaker session gives us execution role + region
sess = sagemaker.Session()
role = get_execution_role()
print(f"SageMaker region : {sess.boto_region_name}")
print(f"Execution role   : ...{role[-30:]}")

# OpenAI key loaded securely - never hardcoded
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key and press Enter: ")

# Build the client once; it reads OPENAI_API_KEY from the environment automatically
client = OpenAI()
print("OpenAI client ready.")

# The five intent categories we classify every customer query into
INTENT_CATEGORIES = [
    "account_inquiry",
    "card_services",
    "loans",
    "investments",
    "general",
]
print("Intent categories:", INTENT_CATEGORIES)

## What Are We Building Today?

Before we write a single new prompt technique, let me show you the exact problem
we are solving.

A Barclays contact centre receives thousands of queries per day. Before a query
reaches a specialist agent, it needs to be routed. Routing requires classification.

Here are three real-sounding queries:

1. `"Can I increase my overdraft limit?"`
2. `"I think there is a fraudulent charge on my Barclaycard."`
3. `"What are the current ISA interest rates?"`

Each one belongs to a different category. If our prompt is too vague, the model
returns inconsistent labels, adds disclaimers, or formats the answer differently
each time - none of which is good for a downstream router.

That is the problem. Let's fix it, step by step.

## Concept 1: Few-Shot Prompting for Intent Classification

I want to show you something that catches people out when they first try to
build a classifier with an LLM.

Watch what happens when I give the model a zero-shot instruction - just a label list,
no examples. The output looks reasonable at first glance, but run the cell a few
times and watch for inconsistencies.

In [ ]:
# NAIVE ZERO-SHOT DEMO - what NOT to do for production classification
# We give the model the category names but no examples of what each one looks like.
# This works sometimes, but breaks on ambiguous queries.

AMBIGUOUS_QUERIES = [
    "I need help with my direct debit",           # account_inquiry or general?
    "What happens if I miss a loan repayment?",   # loans or general?
    "Can I get a new card if mine is damaged?",   # card_services or general?
]

for query in AMBIGUOUS_QUERIES:
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                # No examples - just the category list. The model has to guess.
                "content": (
                    "Classify the following customer query into one of these categories: "
                    + ", ".join(INTENT_CATEGORIES)
                    + ". Reply with the category name only."
                )
            },
            {"role": "user", "content": query}
        ],
        temperature=0.0,
        max_tokens=30
    )
    label = response.choices[0].message.content.strip()
    print(f"Query : {query}")
    print(f"Label : {label}")
    print()

print("Notice: some labels may vary across runs, and the boundary between")
print("'general' and a specific category is unclear without examples.")

### How Few-Shot Prompting Works

```mermaid
graph TD
    ML[messages list] --> SM[system message]
    ML --> UM[user message]
    SM --> E1[Example: statements query - account_inquiry]
    SM --> E2[Example: card declined - card_services]
    SM --> E3[Example: ISA question - investments]
    UM --> LQ[live query from customer]
    SM --> GPT[gpt-4o model]
    UM --> GPT
    GPT --> OUT[classification result]
```

**The core idea**: instead of describing the categories in words, I show the model
examples of (query, label) pairs. The model generalizes from the pattern.

Three things worth knowing:

1. **Order matters (recency bias)**: the model pays more attention to the last
   example in the list. Put the most representative example last.
2. **Label balance**: if 4 of your 5 examples show `account_inquiry`, the model
   will over-predict that category. Balance your examples across categories.
3. **3 to 5 examples** is the sweet spot for classification tasks. More examples
   cost tokens; fewer leave too much ambiguity.

The examples live in the system message, before the user turn. This keeps them
out of the conversation history so they do not shift as turns accumulate.

In [ ]:
# FULL WORKING DEMO - few-shot intent classifier
# I'm going to show you the exact pattern we will use throughout the course.

# Five labeled examples - one per category, balanced, most representative last
FEW_SHOT_EXAMPLES = [
    ("I would like to see my last three statements",         "account_inquiry"),
    ("My Barclaycard was declined at the supermarket",       "card_services"),
    ("What is the early repayment charge on my loan?",       "loans"),
    ("I want to open a stocks and shares ISA",               "investments"),
    ("How do I update my contact details?",                  "general"),
]

def _build_few_shot_system_prompt(examples: list, categories: list) -> str:
    """
    Build a system prompt that contains labeled examples.
    examples: list of (query_text, label) tuples
    categories: list of valid category strings
    """
    # Start with a clear task description and the list of valid categories
    lines = [
        "You are a Barclays customer query classifier.",
        "Classify each customer query into exactly one of these categories:",
        ", ".join(categories),
        "",
        "Here are examples of how to classify queries:",
        "",
    ]
    # Append each labeled example in a consistent (Query / Category) format
    # Consistent formatting helps the model recognize the pattern reliably
    for query_text, label in examples:
        lines.append(f"Query: {query_text}")
        lines.append(f"Category: {label}")
        lines.append("")

    lines.append("Now classify the following query. Reply with the category name only.")
    return "\n".join(lines)


def classify_intent(query: str) -> str:
    """
    Classify a customer query into one of the five Barclays intent categories.
    Returns the category name as a string.
    """
    system_prompt = _build_few_shot_system_prompt(FEW_SHOT_EXAMPLES, INTENT_CATEGORIES)

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": query}
        ],
        temperature=0.0,   # deterministic - we want the same answer every time
        max_tokens=20      # category name is at most 3 words
    )
    return response.choices[0].message.content.strip().lower()


# Test the classifier on the same ambiguous queries from the naive demo
print("Few-shot classifier results:")
print()
for query in AMBIGUOUS_QUERIES:
    label = classify_intent(query)
    print(f"Query : {query}")
    print(f"Label : {label}")
    print()

## Lab 1: Build Your Own Few-Shot Classifier

Now it's your turn. I want you to write a few-shot system prompt that classifies
queries and test it on three new examples.

**Your task**: Complete the `my_classify_intent()` function below. You will write
your own set of labeled examples and build the system prompt yourself.

**Stretch** (if you finish early): Add a sixth category called `"complaint"` and
extend your examples to cover it. Test whether the model correctly distinguishes
between a complaint (`"I am extremely unhappy with this service"`) and a general query.

**Time**: 15-20 minutes

In [ ]:
# Lab 1 SOLUTION: Build Your Own Few-Shot Classifier
# All 4 placeholders filled with working code.

# Step 1: write your own 5 labeled examples - one per category
# SOLUTION: Balance is exactly 1 example per category. The MOST representative
# example ("general" - the catch-all) is last because the model exhibits recency
# bias and weighs the last example most heavily.
MY_EXAMPLES = [
    ("How much money is in my current account right now?",   "account_inquiry"),
    ("My contactless payment was declined at the till",      "card_services"),
    ("How long does a personal loan application take?",      "loans"),
    ("Can I move money from my ISA into a new fund?",        "investments"),
    ("Where do I update my postal address?",                 "general"),
]

# Step 2: build the system prompt using your examples
# SOLUTION: Reuse the helper from the demo - keeps formatting identical so the
# model sees the same Query/Category template it was trained on in earlier cells.
my_system_prompt = _build_few_shot_system_prompt(MY_EXAMPLES, INTENT_CATEGORIES)

def my_classify_intent(query: str) -> str:
    """
    Classify a customer query using your own few-shot examples.
    Returns the category name as a lowercase string.
    """
    # Step 3: deterministic + tight token budget
    # SOLUTION: temperature=0.0 makes the call reproducible. max_tokens=20 is
    # plenty since the longest valid label is "account_inquiry" (~3 tokens).
    my_response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": my_system_prompt},
            {"role": "user",   "content": query}
        ],
        temperature=0.0,
        max_tokens=20
    )

    # Step 4: normalize the output before returning
    # SOLUTION: .strip() removes accidental whitespace; .lower() ensures the
    # returned string matches INTENT_CATEGORIES exactly (the model sometimes
    # capitalizes the first letter even when shown lowercase examples).
    return my_response.choices[0].message.content.strip().lower()


# Test queries - these should each land in a different category
TEST_QUERIES = [
    "I want to apply for a personal loan of 10,000 pounds",
    "Can I get a replacement PIN for my debit card?",
    "How do I set up a standing order?",
]

# Verification
results = []
for q in TEST_QUERIES:
    label = my_classify_intent(q)
    results.append(label)
    print(f"Query : {q}")
    print(f"Label : {label}")
    print()

if all(r in INTENT_CATEGORIES for r in results):
    print("[OK] Lab 1 complete - all labels are valid category names!")
else:
    invalid = [r for r in results if r not in INTENT_CATEGORIES]
    print(f"[ ] Some labels are not in INTENT_CATEGORIES: {invalid}")
    print("    Check your examples and system prompt.")


In [ ]:
# Safety-net: if my_classify_intent is not working yet, run this cell so the
# rest of the notebook still works. Skip this cell if Lab 1 is complete.

def _lab1_fallback_needed():
    try:
        return my_classify_intent("test query") not in INTENT_CATEGORIES
    except Exception:
        return True

if _lab1_fallback_needed():
    print("Using safety-net classify_intent for downstream cells.")
    my_classify_intent = classify_intent
    print("my_classify_intent -> classify_intent (demo version)")
else:
    print("Lab 1 complete - no safety-net needed.")


## Concept 2: Structured Outputs with json_schema

The classifier we just built returns a plain string - the category name.
That is fine for routing, but in production we usually want more:

- A confidence level so a downstream router can decide to escalate low-confidence queries
- A short rationale so the agent can explain the classification decision to a supervisor
- A machine-parseable format so the router does not have to parse free text

Let me show you the naive approach first: json mode without a schema.
The model returns JSON, but there are no guarantees about which fields it includes.


In [ ]:
# NAIVE PATTERN - json_object mode
# This tells the model to return JSON, but it does NOT enforce a schema.
# The model decides what fields to include - they vary between calls.

naive_schema_response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {
            "role": "system",
            "content": (
                "You are a Barclays query classifier. "
                "Return a JSON object with the classification result. "
                "Categories: " + ", ".join(INTENT_CATEGORIES)
            )
        },
        {"role": "user", "content": "I want to check my account balance."}
    ],
    response_format={"type": "json_object"},   # tells model to return JSON
    temperature=0.0,
    max_tokens=100
)

raw = naive_schema_response.choices[0].message.content
parsed = json.loads(raw)
print("json_object mode output:")
print(json.dumps(parsed, indent=2))
print()
print("Notice: field names are unpredictable. Might be 'intent', 'category',")
print("'classification', or something else entirely. This breaks downstream parsers.")


### Anatomy of a json_schema Structured Output

```mermaid
graph TD
    RF[response_format] --> T[type: json_schema]
    RF --> JS[json_schema]
    JS --> N[name: classification_result]
    JS --> ST[strict: True]
    JS --> SC[schema]
    SC --> I[intent - enum of 5 categories]
    SC --> CF[confidence - enum: high, medium, low]
    SC --> RA[rationale - string]
    SC --> RQ[required: all three fields]
    SC --> AP[additionalProperties: False]
```

**json_schema strict mode** (available in gpt-4o-2024-08-06 and later, which is what
the `gpt-4o` alias resolves to) enforces the schema on every token the model generates.
The model cannot add a field that is not in `properties`. It cannot omit a field that
is in `required`.

Three rules to follow every time:

1. Set `"strict": True` inside the `json_schema` object
2. Set `"additionalProperties": False` on every object in the schema
3. List every field you want in `"required"` - the model will always populate them

`enum` is fully supported in strict mode - ideal for constraining the intent and
confidence fields to a fixed set of values.


In [ ]:
# FULL WORKING DEMO - structured output with json_schema strict mode

# Define the schema we want for every classification response.
# Every field is required, enum constrains valid values, additionalProperties is False.
CLASSIFICATION_SCHEMA = {
    "type": "object",
    "properties": {
        "intent": {
            "type": "string",
            "enum": INTENT_CATEGORIES,      # only valid category names allowed
            "description": "The intent category of the customer query."
        },
        "confidence": {
            "type": "string",
            "enum": ["high", "medium", "low"],
            "description": "Model confidence in the classification."
        },
        "rationale": {
            "type": "string",
            "description": "One sentence explaining why this category was chosen."
        }
    },
    "required": ["intent", "confidence", "rationale"],
    "additionalProperties": False           # required for strict mode
}

def classify_with_schema(query: str) -> dict:
    """
    Classify a customer query and return a structured dict with guaranteed fields.
    Returns: {"intent": str, "confidence": str, "rationale": str}
    """
    system_prompt = _build_few_shot_system_prompt(FEW_SHOT_EXAMPLES, INTENT_CATEGORIES)

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": query}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name":   "classification_result",  # arbitrary name for the schema
                "strict": True,                     # enforce the schema on every token
                "schema": CLASSIFICATION_SCHEMA
            }
        },
        temperature=0.0,
        max_tokens=150
    )

    # The model always returns valid JSON matching our schema when strict=True
    raw = response.choices[0].message.content
    return json.loads(raw)


# Test on several queries
demo_queries = [
    "I want to apply for a personal loan.",
    "My Barclaycard statement looks wrong.",
    "How do I close my savings account?",
]

print("Structured classifier results:")
print()
for q in demo_queries:
    result = classify_with_schema(q)
    print(f"Query     : {q}")
    print(f"Intent    : {result['intent']}")
    print(f"Confidence: {result['confidence']}")
    print(f"Rationale : {result['rationale']}")
    print()


## Lab 2: Implement the Schema Classifier

Time to build the structured classifier yourself. You will define a schema and
wire it into the API call.

**Your task**: Complete `my_classify_with_schema()` below. The function must return
a dict with the three required fields.

**Stretch** (if you finish early): Add a fourth field `"escalate"` (boolean) to
your schema. Set it to True when the confidence is "low". Test it with an ambiguous
query like "I have a problem with Barclays."

**Time**: 15-20 minutes


In [ ]:
# Lab 2 SOLUTION: Implement the Schema Classifier
# All 3 placeholders filled with working code.

# Step 1: define the schema dict
# SOLUTION: Strict mode has three non-negotiables - (1) every property listed
# in "required", (2) "additionalProperties": False on every object, (3) enums
# wherever the field has a fixed set of valid values (intent, confidence).
# Without (3) the model is free to invent strings even though the schema validates.
MY_SCHEMA = {
    "type": "object",
    "properties": {
        "intent": {
            "type": "string",
            "enum": INTENT_CATEGORIES,           # enum prevents the model from inventing categories
            "description": "The intent category of the customer query."
        },
        "confidence": {
            "type": "string",
            "enum": ["high", "medium", "low"],   # enum on confidence too - downstream router relies on these exact strings
            "description": "Model confidence in the classification."
        },
        "rationale": {
            "type": "string",
            "description": "One sentence explaining why this category was chosen."
        }
    },
    "required": ["intent", "confidence", "rationale"],
    "additionalProperties": False                # mandatory for strict mode - guarantees no surprise fields
}

def my_classify_with_schema(query: str) -> dict:
    """
    Classify a customer query and return a dict guaranteed to have:
      - "intent": one of INTENT_CATEGORIES
      - "confidence": "high", "medium", or "low"
      - "rationale": short explanation string
    """
    system_prompt = _build_few_shot_system_prompt(FEW_SHOT_EXAMPLES, INTENT_CATEGORIES)

    # Step 2: wire the schema into the API call
    # SOLUTION: response_format wraps the schema. "name" is an arbitrary
    # identifier (useful for logging/debugging). "strict": True is the magic
    # flag that enforces the schema on every generated token - without it the
    # schema is only a hint and the model can drift.
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": query}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name":   "my_classification_result",
                "strict": True,
                "schema": MY_SCHEMA
            }
        },
        temperature=0.0,
        max_tokens=150
    )

    # Step 3: parse and return
    # SOLUTION: With strict=True the content is guaranteed valid JSON matching
    # MY_SCHEMA, so json.loads is safe without try/except. In production code
    # I would still wrap this in try/except to guard against transport errors.
    return json.loads(response.choices[0].message.content)


# Verification
lab2_test_queries = [
    "I need a new debit card, mine expired.",
    "What are my options for investing my savings?",
    "Can I borrow 5000 pounds over 3 years?",
]

all_valid = True
for q in lab2_test_queries:
    result = my_classify_with_schema(q)
    required_keys = {"intent", "confidence", "rationale"}
    has_all_keys = required_keys.issubset(result.keys())
    valid_intent = result.get("intent") in INTENT_CATEGORIES
    print(f"Query     : {q}")
    print(f"Intent    : {result.get('intent', 'MISSING')}")
    print(f"Confidence: {result.get('confidence', 'MISSING')}")
    print(f"Rationale : {result.get('rationale', 'MISSING')}")
    print(f"Keys OK   : {has_all_keys}  Intent valid: {valid_intent}")
    print()
    if not (has_all_keys and valid_intent):
        all_valid = False

if all_valid:
    print("[OK] Lab 2 complete - all responses have required fields and valid intent!")
else:
    print("[ ] Some responses are missing fields or have invalid intent values.")
    print("    Check your schema definition and response_format dict.")


In [ ]:
# Safety-net: if my_classify_with_schema is not working yet, run this cell so
# later cells still work. Skip this cell if Lab 2 is complete.

def _lab2_fallback_needed():
    try:
        r = my_classify_with_schema("test query")
        return not ({"intent", "confidence", "rationale"}.issubset(r.keys()))
    except Exception:
        return True

if _lab2_fallback_needed():
    print("Using safety-net classify_with_schema for downstream cells.")
    my_classify_with_schema = classify_with_schema
    print("my_classify_with_schema -> classify_with_schema (demo version)")
else:
    print("Lab 2 complete - no safety-net needed.")


## Concept 3: Iterative Prompt Refinement

Here is something I see all the time: a developer writes a prompt, tests it on
two examples, declares it "good enough", and ships it. Three days later the router
is misclassifying 20% of queries in production.

The problem is not the model. The problem is that the developer never ran a
systematic test.

Let me show you a prompt that looks reasonable but scores poorly on a broader
test battery.


In [ ]:
# WEAK PROMPT DEMO - a prompt that looks fine on one or two examples
# but fails on a structured test battery

WEAK_SYSTEM_PROMPT = (
    "You are a helpful assistant. "
    "Classify the customer query. "
    "Return a JSON object with intent and confidence."
)

# A test battery of 5 queries that cover edge cases and category boundaries
TEST_BATTERY = [
    {"query": "I want to pay off my credit card balance early.", "expected": "card_services"},
    {"query": "What documents do I need to apply for a mortgage?",  "expected": "loans"},
    {"query": "Can I open a junior ISA for my child?",              "expected": "investments"},
    {"query": "My direct debit was taken twice this month.",         "expected": "account_inquiry"},
    {"query": "How do I find my nearest Barclays branch?",          "expected": "general"},
]

print("Testing the weak prompt on the test battery:")
print()

weak_correct = 0
for item in TEST_BATTERY:
    r = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": WEAK_SYSTEM_PROMPT},
            {"role": "user",   "content": item["query"]}
        ],
        response_format={"type": "json_object"},
        temperature=0.0,
        max_tokens=80
    )
    raw = json.loads(r.choices[0].message.content)
    # The field name is not guaranteed - the model may use 'intent', 'category', etc.
    predicted = raw.get("intent") or raw.get("category") or raw.get("classification") or "unknown"
    predicted = str(predicted).lower().strip()
    correct = predicted == item["expected"]
    if correct:
        weak_correct += 1
    status = "[OK]" if correct else "[ ]"
    print(f"{status} Query    : {item['query']}")
    print(f"   Expected : {item['expected']}")
    print(f"   Got      : {predicted}")
    print()

print(f"Weak prompt score: {weak_correct}/{len(TEST_BATTERY)}")
print("Watch how the score changes after we apply iterative refinement.")


### The Iterative Prompt Refinement Loop

```mermaid
graph TD
    A[write initial prompt] --> B[run test battery - 5 queries]
    B --> C[score 3 dimensions - 1 to 5 each]
    C --> D{overall score above 4?}
    D -- Yes --> G[best prompt found]
    D -- No --> E[identify lowest-scoring dimension]
    E --> F[revise prompt for that dimension only]
    F --> B
```

**The three scoring dimensions** (each scored 1-5):

| Dimension | What it measures |
|-----------|-----------------|
| format | Does the output match the expected structure every time? |
| relevance | Is the predicted intent correct for this query? |
| constraint_adherence | Does the output respect the category list and schema? |

A **numeric scorecard** is more useful than a pass/fail check because it tells you
which dimension to fix next. If format is 5/5 but relevance is 2/5, the prompt
wording is the problem - not the output structure.

The key discipline: **fix ONE dimension per iteration**. Changing everything at once
makes it impossible to know which change helped.


In [ ]:
# FULL WORKING DEMO - iterative prompt refinement with a numeric scorecard

def score_prompt(system_prompt: str, test_battery: list) -> dict:
    """
    Run a system prompt against the test battery and return a scorecard.
    Returns: {
        "format": float (1-5),
        "relevance": float (1-5),
        "constraint_adherence": float (1-5),
        "overall": float (average of the three),
        "details": list of per-query result dicts
    }
    """
    format_scores      = []
    relevance_scores   = []
    constraint_scores  = []
    details            = []

    for item in test_battery:
        r = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": item["query"]}
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name":   "classification_result",
                    "strict": True,
                    "schema": CLASSIFICATION_SCHEMA
                }
            },
            temperature=0.0,
            max_tokens=150
        )

        try:
            parsed = json.loads(r.choices[0].message.content)
            # Format score: did we get all required fields?
            has_fields = all(k in parsed for k in ["intent", "confidence", "rationale"])
            fmt_score  = 5 if has_fields else 1

            # Relevance score: does the predicted intent match expected?
            predicted  = parsed.get("intent", "")
            rel_score  = 5 if predicted == item["expected"] else 1

            # Constraint adherence: is the intent value in our allowed list?
            con_score  = 5 if predicted in INTENT_CATEGORIES else 1

        except (json.JSONDecodeError, KeyError):
            fmt_score = rel_score = con_score = 1
            parsed    = {}

        format_scores.append(fmt_score)
        relevance_scores.append(rel_score)
        constraint_scores.append(con_score)
        details.append({
            "query":    item["query"],
            "expected": item["expected"],
            "got":      parsed.get("intent", "ERROR"),
            "format":   fmt_score,
            "relevance":  rel_score,
            "constraint": con_score
        })

    avg = lambda lst: sum(lst) / len(lst)
    return {
        "format":              avg(format_scores),
        "relevance":           avg(relevance_scores),
        "constraint_adherence": avg(constraint_scores),
        "overall":             avg(format_scores + relevance_scores + constraint_scores),
        "details":             details
    }


def print_scorecard(label: str, scorecard: dict) -> None:
    """Print a readable scorecard."""
    print(f"=== {label} ===")
    print(f"  Format              : {scorecard['format']:.1f}/5")
    print(f"  Relevance           : {scorecard['relevance']:.1f}/5")
    print(f"  Constraint adherence: {scorecard['constraint_adherence']:.1f}/5")
    print(f"  Overall             : {scorecard['overall']:.1f}/5")
    print()


# Iteration 0 - the weak prompt we already tested
prompt_v0 = WEAK_SYSTEM_PROMPT
score_v0  = score_prompt(prompt_v0, TEST_BATTERY)
print_scorecard("Iteration 0 (weak prompt)", score_v0)

# Iteration 1 - add few-shot examples to fix relevance
prompt_v1 = _build_few_shot_system_prompt(FEW_SHOT_EXAMPLES, INTENT_CATEGORIES)
score_v1  = score_prompt(prompt_v1, TEST_BATTERY)
print_scorecard("Iteration 1 (few-shot added)", score_v1)

# Iteration 2 - add explicit constraint reminder to fix any constraint adherence issues
prompt_v2 = prompt_v1 + (
    "\n\nIMPORTANT: you MUST use only these exact category names: "
    + ", ".join(INTENT_CATEGORIES)
    + ". Any other value is an error."
)
score_v2  = score_prompt(prompt_v2, TEST_BATTERY)
print_scorecard("Iteration 2 (constraint reminder added)", score_v2)

# Pick the best prompt based on overall score
best_score  = max([score_v0, score_v1, score_v2], key=lambda s: s["overall"])
best_prompt = [prompt_v0, prompt_v1, prompt_v2][
    [score_v0, score_v1, score_v2].index(best_score)
]

print(f"Best overall score: {best_score['overall']:.1f}/5")
print("best_prompt set to the iteration with the highest overall score.")


## Lab 3: Run Your Own Refinement Cycle (Tier 2 - Hard)

This is the hard lab for Day 2. There are fewer hints. You will need to
think through the steps yourself.

**Your task**: Design a new system prompt for the intent classifier, run it
through at least three refinement iterations using `score_prompt()`, and
improve the overall score compared to where you started.

**What you need to do**:
- Write a starting prompt (different from the ones above - try a different
  instruction style, or different examples)
- Run it through `score_prompt()` with `TEST_BATTERY`
- Look at the weakest scoring dimension in the scorecard
- Revise the prompt to address that specific dimension
- Repeat for at least two more iterations
- Store your final best prompt in `my_best_prompt`

**What you are NOT given**:
- The exact shape of the loop (you decide how to structure it)
- Which dimension to fix first (the scorecard tells you)
- What changes to make (that is the real prompt engineering work)

**Stretch** (if you finish early): Write three additional test queries that
you believe are genuinely ambiguous, add them to the battery, and run the
same refinement loop. Does your best prompt handle the new queries well?

**Time**: 25-35 minutes


In [ ]:
# Lab 3 SOLUTION: Iterative Refinement Cycle (Tier 2 - Hard)
# 3 iterations + my_best_prompt selected by overall score.

# SOLUTION strategy: I'm following the discipline from the lecture:
# (1) Make iteration 0 deliberately under-specified so I can SEE which dimension
#     is weak. (2) Fix exactly ONE dimension per iteration. (3) Pick the best
#     prompt by overall score, breaking ties in favor of the simpler prompt.

# Iteration 0 - role-only instruction. No examples, no category list.
# This isolates RELEVANCE as the weakest dimension - the model has to guess
# both the categories and the format from a one-line instruction.
my_prompt_v0 = (
    "You work in Barclays customer service. "
    "Decide what kind of query this is and explain your reasoning."
)

# SOLUTION: establish the baseline
score0 = score_prompt(my_prompt_v0, TEST_BATTERY)
print_scorecard("My iteration 0 (role only)", score0)

# Iteration 1 - relevance was the weakest dimension, so I add the category list
# AND 3 worked examples covering the most-confused boundaries (card vs general,
# loans vs general, account-close vs general).
# SOLUTION: Note I deliberately do NOT add a constraint reminder yet. The whole
# point of "fix ONE dimension per iteration" is that you can attribute the score
# delta to a specific change. If I changed two things at once and the score
# went up, I would not know which change helped.
my_prompt_v1 = (
    "You work in Barclays customer service. Classify the query into exactly one of: "
    + ", ".join(INTENT_CATEGORIES)
    + ".\n\nExamples:\n"
    + "Query: I want to dispute a transaction on my Barclaycard\nCategory: card_services\n"
    + "Query: What is the maximum loan amount I can apply for?\nCategory: loans\n"
    + "Query: How do I close my account?\nCategory: general\n"
)
score1 = score_prompt(my_prompt_v1, TEST_BATTERY)
print_scorecard("My iteration 1 (categories + examples)", score1)

# Iteration 2 - relevance is up; if constraint adherence is now the weakest,
# pin the model to the exact strings. Even with json_schema enum the wording
# matters because the model's internal reasoning still influences which enum it picks.
my_prompt_v2 = my_prompt_v1 + (
    "\nReturn ONLY one of the listed category names. "
    "Do not invent new categories. Do not combine categories."
)
score2 = score_prompt(my_prompt_v2, TEST_BATTERY)
print_scorecard("My iteration 2 (constraint reminder)", score2)


# SOLUTION: pick the best prompt by overall score
# In production you also want to track cost (token count) and latency, but
# for this lab "highest overall score" is enough. max() is stable so on ties
# it returns the first match - which conveniently is the simpler, cheaper prompt.
candidates = [(my_prompt_v0, score0), (my_prompt_v1, score1), (my_prompt_v2, score2)]
my_best_prompt = max(candidates, key=lambda c: c[1]["overall"])[0]

# Verification
if my_best_prompt is not None:
    final_score = score_prompt(my_best_prompt, TEST_BATTERY)
    print_scorecard("My best prompt", final_score)
    if final_score["overall"] >= 3.5:
        print("[OK] Lab 3 complete - overall score >= 3.5/5!")
    else:
        print("[ ] Overall score below 3.5 - keep refining!")
else:
    print("[ ] my_best_prompt is still None - complete your refinement loop above.")


In [ ]:
# Safety-net: if my_best_prompt is not set, fall back to prompt_v2
# so the wrap-up cell works. Skip this if Lab 3 is complete.

if my_best_prompt is None:
    print("Using safety-net my_best_prompt = prompt_v2.")
    my_best_prompt = prompt_v2
    print("my_best_prompt set to prompt_v2 (demo version).")
else:
    print("Lab 3 complete - no safety-net needed.")


## Topic 4 Wrap-Up: What We Built Today

Let me recap what you just built and connect it to the bigger picture.

**What you built today**:
- `classify_intent(query)` - a few-shot intent classifier that maps customer
  queries to one of five Barclays categories
- `classify_with_schema(query)` - the same classifier with guaranteed JSON output
  fields (intent, confidence, rationale) using json_schema strict mode
- A numeric scorecard and iterative refinement loop that measures three dimensions
  (format, relevance, constraint adherence) and guides targeted improvements

**The Barclays Product Knowledge Assistant so far**:

  customer query
        |
        v
  [classify_intent / classify_with_schema]  <- built this topic
        |
        v
  {"intent": "loans", "confidence": "high", "rationale": "..."}
        |
        v
  [route to product specialist or answer directly]

**Key takeaways**:
- Few-shot examples dramatically improve classification accuracy on ambiguous queries
- json_schema strict mode guarantees field names and value types - critical for any
  downstream code that parses the output
- Systematic test batteries + scorecards beat ad-hoc testing every time

**Connection to Topic 5 (Conversation Memory)**:
In Topic 5 we build a multi-turn chatbot. `classify_intent()` and `classify_with_schema()`
will be called at the start of each turn to decide whether to retrieve product information
or handle the query with general knowledge. We carry these functions forward directly.

**Homework Extensions** (optional):
1. Add a "complaint" category to the schema and extend the few-shot examples.
   Test whether the model correctly distinguishes between complaints and general queries.
2. Increase the test battery to 20 queries and track how the overall score changes
   across 5 refinement iterations. Plot the scores using matplotlib.
3. Replace the simple 1/5 binary scoring with a call to GPT-4o-as-judge that
   rates relevance on a 1-5 scale with reasoning. Compare the automated judge
   scores to your manual assessment.
